# 4.5 Het residu fitten — ruis, of een vergeten mechanisme

De rest van deze les fitte verdelingen op ruwe data. Het meest waardevolle om een verdeling op te fitten is echter **wat een model niet kon verklaren**: fit een model, trek het af, en vraag aan het residu tot welke familie het behoort. De vraag die dit notebook beantwoordt is: *hoe weet je wanneer een model klaar is?* We bouwen voort op de fitter, de twee winnaars en `PlotFits` uit [04.2](04.2-long-tail%20jeffrey.ipynb), en de interpretatie van overschrijdingskansen bij uitbijters uit 04.2 en [04.3](04.3-penguins%20jeffrey.ipynb).

- Een **symmetrisch residu uit een plausibele familie**, zonder patroon over de tijd, betekent dat het model alle verklaarbare structuur heeft geëxtraheerd. Wat overblijft is pure ruis, en je moet stoppen met fitten.
- Een **residu met structuur over de tijd** — een sprong, een trendmatige afwijking, een bult — is helemaal geen verdelingsprobleem. Het is een niet-gemodelleerd mechanisme, en *wanneer* het begint is meestal de eigenlijke wetenschappelijke ontdekking.

Dit notebook doorloopt die leercyclus op openbare Nederlandse COVID-cijfers, bewust geen chatdata: de methodologische cyclus is immers het overdraagbare deel. Het model is een eenvoudige rechte lijn. Op het tijdvenster waar de aanname standhoudt, is het residu ruis en de fitter bevestigt dat. Op het volledige tijdvenster laat diezelfde rechte lijn een residu achter met een duidelijke vorm. Die vorm benoemt exact wat het model mist en dateert het moment van verandering. Dát is waar "het fitten van het residu" voor dient: het vertelt je of je model klaar is, en zo niet, wat voor soort term je moet toevoegen. Het toevoegen van die term is de kern van les 5.


In [ ]:
import numpy as np

from goad_toolkit.analytics import DistributionFitter, FitResult, fit_table
from goad_toolkit.config import DataConfig, FileConfig
from goad_toolkit.dataprocessor import CovidDataProcessor
from goad_toolkit.models import linear_model, mse, train_model
from goad_toolkit.visualizer import (
    ComparePlot,
    FitPlotSettings,
    PlotFits,
    PlotSettings,
    QQPlot,
    ResidualPlot,
)

## 4.5.1 De data, en de pipeline die deze construeert

Dagelijkse Nederlandse COVID-cijfers, oktober 2020 tot juni 2021: positieve testen en sterfgevallen per dag. `CovidDataProcessor` downloadt het ruwe bronbestand eenmalig en voert er een `Pipeline` overheen uit — exact het type object dat les 1 introduceerde, opgebouwd uit heldere stappen:

1. `DiffValues` op `deaths` — de bron rapporteert het *cumulatieve* aantal; het verschil is het aantal overledenen op die specifieke dag.
2. `ShiftValues(period=-14)` → `deaths_shifted` — een overlijden volgt gemiddeld zo'n twee weken na een positieve test. De sterfgevallen die veertien dagen *later* zijn geregistreerd, worden daarom teruggeschoven naar de rij van vandaag, naast de testen die ertoe hebben geleid. Dat is de tijdsvertraging (*lag*) die een model tussen beide variabelen vereist; zonder deze verschuiving zou de lijn worden gefit op paren die niets met elkaar te maken hebben.
3. `SelectDataRange` — het afgebakende tijdvenster.
4. `RollingAvg(window=7)` op beide — rapportage kent een wekelijkse cyclus (minder registraties in het weekend) die geen onderdeel is van het ziekteverloop zelf.
5. `ZScaler` → `*_zscore` — overlijdens betreffen tientallen per dag, testen tienduizenden. Om te zien of beide tijdreeksen dezelfde *vorm* hebben, brengen we ze eerst naar dezelfde schaal.


In [ ]:
processor = CovidDataProcessor(FileConfig(), DataConfig())
data = processor.process()
print(f"{len(data)} days, {data.index.min().date()} to {data.index.max().date()}")
data[["positivetests", "deaths", "deaths_shifted", "deaths_shifted_zscore", "positivetests_zscore"]].head(3)

In [ ]:
fig, ax = ComparePlot(PlotSettings(figsize=(11, 4), xlabel="date", ylabel="z-score",
                                   title="Positive tests and (lagged) deaths, on one scale")).plot(
    data=data, x="date", y1="positivetests_zscore", y2="deaths_shifted_zscore",
)

Gedurende de herfst en winter volgen de twee curven elkaar nauwgezet: wanneer het aantal testen stijgt, stijgt het aantal overlijdens twee weken later met een evenredig percentage. Dat is exact de aanname die een rechte lijn maakt — een constante ratio van sterfgevallen per positieve test. Het is waardevol om dat vooraf hardop uit te spreken vóórdat je fit, want de tweede helft van de grafiek verraadt al waar die aanname ophoudt te gelden: vanaf maart lopen de testen weer op, maar de overlijdens volgen niet.


## 4.5.2 Een model op het tijdvenster waar de aanname standhoudt

Het vaccineren in Nederland begon op 6 januari 2021. Fit de rechte lijn op de dagen vóór die datum, waar de ratio stabiel oogt: `deaths_shifted ≈ a · positivetests + b`, met goad's modelleringsset — `linear_model` is de wiskundige vorm, `mse` is de verliesfunctie (*loss*), en `train_model` vindt de optimale parameters `a` en `b`. Bekijk vervolgens wat de lijn níet kon verklaren.


In [ ]:
VACCINATION_START = "2021-01-06"
before = data.index < VACCINATION_START

x = data["positivetests"].to_numpy()
y = data["deaths_shifted"].to_numpy()

params_before = train_model(x[before], y[before], linear_model, mse, [0.01, 1.0],
                            bounds=[(0, 1.0), (0, None)])
print(f"deaths ≈ {params_before[0]:.4f} · tests + {params_before[1]:.1f}")

early = data[before].copy()
early["predicted"] = linear_model(x[before], params_before)
early["residual"] = early["deaths_shifted"] - early["predicted"]

fig, ax = ComparePlot(PlotSettings(figsize=(11, 4), xlabel="date", ylabel="deaths",
                                   title="Before vaccination: a straight line in positive tests")).plot(
    data=early, x="date", y1="deaths_shifted", y2="predicted",
)

In [ ]:
fig, ax = ResidualPlot(PlotSettings(figsize=(11, 3.5), xlabel="date", ylabel="error",  # ty: ignore[invalid-argument-type]
                                    title="What the line did not explain, before vaccination")).plot(
    data=early, x="date", y="residual", date=VACCINATION_START, datelabel="vaccination starts",
    interval=2,
)

Golven van een week of twee aan weerszijden van de nullijn — wat een 7-daagse afvlakking van grillige dagtotalen nu eenmaal achterlaat — en geen systematische drift over het tijdvenster. Kijk echter kritisch naar de laatste twee weken: de lijn schiet tekort bij de winterpiek, met een aaneenschakeling van positieve fouten tot aan de rand van het venster. Houd dat in gedachten; het is de eerste hint van de vorm die het volledige venster dadelijk zal openbaren. De eerste controle is verder helder, dus door naar de tweede: de kansverdeling van de foutterm zelf. Dit is waar `DistributionFitter` haar waarde bewijst in een modelleringscyclus: **toets het residu**. Fit elke continue familie en lees de tabel op vorm en overeenstemming, niet op één enkele winnaar.


In [ ]:
fitter = DistributionFitter(seed=42)
fits_before = fitter.fit(early["residual"].to_numpy(), discrete=False)
fit_table(fits_before)[["distribution", "log_likelihood", "ks_pvalue", "best_likelihood", "best_ks"]]

In [ ]:
fig = PlotFits(PlotSettings(figsize=(12, 4), xlabel="error", ylabel="density",
                            title="Before vaccination: the residual, top three fits")).plot(
    data=early["residual"].to_numpy(), fit_results=fits_before,
    fitplotsettings=FitPlotSettings(bins=25, max_fits=3),
)

Gecentreerd rond nul, en een half dozijn families binnen enkele log-likelihood punten van elkaar — beta, weibull, gamma, lognormaal, skew-normaal, normaal — waarvan er geen enkele wordt verworpen door de KS-toets. De enige familie die *wél* wordt verworpen, `uniform`, wordt om de juiste reden verworpen: de fouten concentreren zich rond nul in plaats van gelijkmatig uitgesmeerd te zijn. Wanneer de winnaar niet betekenisvol voorligt, kan de data de families niet van elkaar onderscheiden, en dát is exact hoe pure ruis eruitziet. Lees dit als een gunstig eindoordeel over het model: op dit tijdvenster vertelt één vaste ratio tussen overlijdens en testen nagenoeg het hele verhaal.


## 4.5.3 Hetzelfde model op het volledige tijdvenster

Fit nu exact dezelfde rechte lijn over alle 232 dagen. Niets in het model weet dat er inmiddels een vaccin bestaat.


In [ ]:
params_full = train_model(x, y, linear_model, mse, [0.01, 1.0], bounds=[(0, 1.0), (0, None)])
data["linear"] = linear_model(x, params_full)
data["residual_linear"] = data["deaths_shifted"] - data["linear"]
print(f"deaths ≈ {params_full[0]:.4f} · tests + {params_full[1]:.1f}   (mse {mse(y, data['linear']):.0f})")

fig, ax = ResidualPlot(PlotSettings(figsize=(11, 4), xlabel="date", ylabel="error",
                                    title="The full window: a residual with a shape")).plot(
    data=data, x="date", y="residual_linear", date=VACCINATION_START,
    datelabel="vaccination starts", interval=1,
)

In [ ]:
fits_full = fitter.fit(data["residual_linear"].to_numpy(), discrete=False)
fit_table(fits_full)[["distribution", "log_likelihood", "ks_pvalue", "best_likelihood", "best_ks"]].head(4)

Twee dingen zijn veranderd, en het zijn twee kanten van dezelfde medaille. De residu-plot heeft nu **een duidelijke vorm**: positief tijdens de wintergolf, en vanaf februari een aanhoudende reeks negatieve residuen die nooit meer terugkeert — de lijn voorspelt maand na maand aanzienlijk méér doden dan er in werkelijkheid vallen. En de fit-tabel geeft het op: elke verdelingsfamilie wordt verworpen, en de familie die "wint" is `uniform` — wat een fitter aangeeft wanneer fouten verspreid liggen langs een trend in plaats van willekeurig verdeeld rond nul.

Dat is geen probleem van kansverdelingen. Het residu vertelt je exact wat het model mist, en grofweg vanaf wanneer: **na de start van het vaccineren leidt hetzelfde aantal positieve testen tot substantieel minder overlijdens.** Eén enkele vaste ratio kan dat per definitie niet uitdrukken. Het residu deed dat zojuist wél.


## 4.5.4 Wat het residu heeft benoemd — en wat nog niet

Maak de balans op van wat twee fits van één rechte lijn hebben aangetoond. Op het venster vóór vaccinatie is het residu ruis: de rechte lijn vertelt het hele verhaal. Op het volledige venster vertoont het residu een specifieke structuur. Het is niet willekeurig verspreid, het is geen eenmalige schok op één dag, en het is geen gestage drift die er vanaf het begin al zat: het is een aaneengesloten reeks negatieve fouten die laat in de winter begint en permanent aanhoudt. De enige aanname van de rechte lijn — een constante ratio tussen testen en doden — verloor haar geldigheid, en het residu dateerde de omslag exact naar de weken volgend op de start van het vaccineren.

Dat is een wetenschappelijke bevinding, en het is dé ontdekking die het gereedschap van deze les kan leveren: **het residu heeft een mechanisme benoemd en gedateerd.** Wat het nog niet heeft gedaan, is wiskundig beschrijven wat die ratio *doet* over de tijd — welke vorm de verandering heeft, en hoe die vorm als term naast de rechte lijn in het model moet worden opgenomen. Het benoemen van de vorm die een model niet kan verklaren, en die vorm als basisfunctie in het model schrijven, is de kern van les 5. [05.3](../lesson5/05.3-covid-modelling%20jeffrey.ipynb) reikt je exact dit residu aan om dat te doen. Ga daarheen met de visuele vorm in je hoofd, niet direct met een formule: de bovenstaande grafiek toont je al globaal wanneer de ratio omsloeg en welke kant op; een heldere zin daarover ís het model.


## 4.5.5 Uitbijters: een punt in de staart is een vraag, geen vonnis

Met een goedgekeurde verdelingsfamilie in handen krijgt de vraag "is dit punt ongebruikelijk?" een eerlijk antwoord: hoeveel overschrijdingskans kent *de familie waaraan je je hebt gecommitteerd* toe aan waarden voorbij dit punt? De standaard z-score-regel beantwoordt een wezenlijk andere vraag — hoe ongebruikelijk zou dit zijn *als de data normaal verdeeld was* — en beide spreken elkaar tegen precies wanneer de data niet normaal verdeeld is.

Het residu waarop je deze vraag moet loslaten is het residu dat je als ruis hebt geaccepteerd — dat van het pre-vaccinatievenster uit §4.5.2 — want een overschrijdingskans berekenen onder een familie die reeds verworpen is, is een getal over gebakken lucht. Toets de dagen op beide manieren.


In [ ]:
residual = early["residual"].to_numpy()
best = next(f for f in fits_before if isinstance(f, FitResult) and f.best_likelihood)

z_flagged = np.abs((residual - residual.mean()) / residual.std()) > 3
tail = np.minimum(best.frozen_dist.cdf(residual), 1 - best.frozen_dist.cdf(residual))
tail_flagged = tail < 0.005  # roughly the two-sided z>3 threshold, but under the fitted family

print(f"z-score rule (assumes normal):        {z_flagged.sum()} day(s) flagged")
print(f"tail probability under {best.distribution:9s}: {tail_flagged.sum()} day(s) flagged")
print(f"the rules disagree on {int((z_flagged != tail_flagged).sum())} day(s)")
for day in early.index[z_flagged | tail_flagged]:
    print(f"  {day.date()}: error {early.loc[day, 'residual']:+.1f}")
fig, ax = QQPlot(PlotSettings(title=f"pre-vaccination residual vs. fitted {best.distribution}",
                              xlabel="theoretical quantile", ylabel="error")).plot(
    data=residual, distribution=best.frozen_dist,
)

De z-score-regel markeert helemaal niets; de gefitte beta-verdeling markeert twee dagen, waarvan één 29 december is — precies in de reeks positieve fouten tijdens de winterpiek die §4.5.2 je al adviseerde om in de gaten te houden. De staart van een residu is waar het volgende probleem van een model zich als eerste openbaart.

Welke kant de twee regels ook op wijzen, de overschrijdingskans van de gefitte familie zelf is het meest verdedigbare getal: het is immers de verdeling waar de rest van de analyse zich al aan heeft gecommitteerd, geen tweede aanname die er via een vuistregel stilzwijgend is binnengesmokkeld. Wat je vervolgens *doet* met een gemarkeerd punt — een invoerfout om te verwijderen, een reële zeldzame dag om te behouden, een nieuw regime waar het model geen term voor heeft, of een verkeerd gekozen verdelingsfamilie — is een inhoudelijke beslissing die proceskennis vereist, geen extra statistiek.

Die domeinkennis ligt bij jou, niet bij de AI-assistent. Vraag de `goad` MCP om het hoofdstuk *Distributions* en lees §5.6, "Outliers, properly" — vijf stappen, en de valkuil waar de z-score instapt:

    goad_get_concept("Distributions")

Neem vervolgens de gemarkeerde dagen uit **je eigen chat** — de grootste residuen uit de event-plot van 03.3, of de dagen die het verst in de staart vallen van de fit uit [04.4](04.4-before-and-after%20jeffrey.ipynb) — en doorloop de vijf stappen hardop. De code berekent de overschrijdingskans; alleen jij weet of die specifieke zaterdag een verjaardag was, een exportfout, of de week waarin iemand lid werd van de groep. Zeg wélke reden het is, en leg uit waarom. Dát is het antwoord om op te schrijven, niet het getal.


## Reflectievragen

1. In je eigen woorden: waarom markeert `DistributionFitter` een *likelihood*-winnaar en een *KS*-winnaar afzonderlijk, en wat vertelde hun overeenstemming (of juist het gebrek daaraan) je bij elk van de twee fits in dit notebook — én bij de samengevoegde fit in [04.3](04.3-penguins%20jeffrey.ipynb)?
2. Het residu in §4.5.3 werd "verworpen door elke familie". Leg uit waarom dat een uitspraak is over het *model*, niet over de verdelingen — en noem het mechanisme waar het naar wees.
3. Het residu dateert de omslag naar het einde van de winter. Noem twee alternatieve mechanismen naast vaccinatie die dezelfde reeks negatieve fouten op dezelfde kalender zouden achterlaten, en welke aanvullende data nodig zou zijn om daartussen te differentiëren.
4. Voor je eigen chat: kies één dag die door de fit uit [04.4](04.4-before-and-after%20jeffrey.ipynb) als ongebruikelijk werd bestempeld, en leg uit waarom de standaard z-score-regel het daar bij het verkeerde eind zou hebben gehad.

---

**Waar dit nu naartoe gaat.** Les 5 start vanuit relaties tussen twee variabelen ([05.1](../lesson5/05.1-relationships%20jeffrey.ipynb)) en keert terug naar dit residu in [05.3](../lesson5/05.3-covid-modelling%20jeffrey.ipynb), waar de vorm die het residu achterliet een volwaardige term in het model wordt — jouw term, gefit en getoetst aan de hoge standaard die dit notebook heeft neergezet.
